**Task01**

In [ ]:
import pandas as pd
import gdown
from sklearn.feature_extraction.text import CountVectorizer
import string
from nltk.tokenize import RegexpTokenizer

In [ ]:
file_id = '1HcwUJs1s365Gg4GvIY0roeEHsdBqJ8mW'
output = 'your_dataset.csv'
gdown.download(f'https://drive.google.com/uc?id={file_id}', output, quiet=True)

df = pd.read_csv(output)
df.columns = ['rating','title','content']
df = df.head(50000)
df = df.drop('title',axis=1)

tokenizer = RegexpTokenizer(r'\w+')
df['content'] = df['content'].astype(str).apply(lambda text: ' '.join(tokenizer.tokenize(text.lower())))

vectorizer = CountVectorizer()
bow_matrix = vectorizer.fit_transform(df['content'])

print("Dimensions of the feature matrix:",bow_matrix.shape)

Dimensions of the feature matrix: (50000, 72111)


**Task02**

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

In [ ]:
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(df['content'])
tfidf_sum = np.asarray(tfidf_matrix.sum(axis=0)).flatten()

feature_names = tfidf_vectorizer.get_feature_names_out()
tfidf_series = pd.Series(tfidf_sum, index=feature_names)

top_words_series = tfidf_series.sort_values(ascending=False).head(15)
top_words = list(top_words_series.items())

print("Top 15 words with highest TF-IDF scores:")
for word, score in top_words:
    print(f"{word}: {score:.4f}")

Top 15 words with highest TF-IDF scores:
the: 4718.8987
it: 2808.4346
and: 2808.3363
to: 2612.3162
of: 2348.3304
this: 2232.1506
is: 2205.5720
in: 1625.1133
was: 1600.5019
book: 1592.9034
that: 1511.6892
for: 1490.3187
you: 1477.4605
not: 1235.6978
but: 1187.8307


**Task03**

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

glove_file_path = '/content/glove.6B.100d.txt'

embeddings = {}
with open(glove_file_path, 'r', encoding='utf8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        vector = np.asarray(values[1:], dtype='float32')
        if len(vector) == 100:
            embeddings[word] = vector

def find_closest_word(vec, embeddings, exclude_words=set()):
    max_similarity = -1
    closest_word = None
    for word, embedding in embeddings.items():
        if word in exclude_words:
            continue
        if embedding.shape == vec.shape:
            similarity = cosine_similarity(vec.reshape(1, -1), embedding.reshape(1, -1))[0][0]
            if similarity > max_similarity:
                max_similarity = similarity
                closest_word = word
    return closest_word

vec_king = embeddings['king']
vec_man = embeddings['man']
vec_woman = embeddings['woman']

analogy_vec = vec_king - vec_man + vec_woman

closest_word = find_closest_word(analogy_vec, embeddings, exclude_words={'king', 'man', 'woman'})

print(f"Result of analogy 'King - Man + Woman': {closest_word}")


Result of analogy 'King - Man + Woman': queen


**Task04**

In [ ]:
!pip install nltk

In [ ]:
import nltk
nltk.download('all')

[nltk_data] Downloading collection 'all'
[nltk_data]    | 
[nltk_data]    | Downloading package abc to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/abc.zip.
[nltk_data]    | Downloading package alpino to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/alpino.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger_eng to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Unzipping
[nltk_data]    |       taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger_ru to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Unzipping
[nltk_data]    |       taggers/averaged_perceptron_tagger_ru.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger_rus to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |  

True

In [ ]:
!pip install gensim
import gensim
from gensim.models import Word2Vec
from nltk.tokenize import word_tokenize
from nltk.corpus import reuters


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 70.9 MB/s eta 0:00:00


In [ ]:
documents = []
for fileid in reuters.fileids():
    raw_text = reuters.raw(fileid)
    tokens = word_tokenize(raw_text.lower())
    documents.append(tokens)

cbow_model = Word2Vec(sentences=documents, vector_size=100, window=5, min_count=5, sg=0)
skipgram_model = Word2Vec(sentences=documents, vector_size=100, window=5, min_count=5, sg=1)

analogy_cbow = cbow_model.wv.most_similar(positive=['government', 'company'], negative=['bank'], topn=5)
analogy_skipgram = skipgram_model.wv.most_similar(positive=['government', 'company'], negative=['bank'], topn=5)

print("CBOW analogy result ('government' - 'bank' + 'company') top 5:")
for word, score in analogy_cbow:
    print(f"{word}: {score:.4f}")

print("\nSkipgram analogy result ('government' - 'bank' + 'company') top 5:")
for word, score in analogy_skipgram:
    print(f"{word}: {score:.4f}")


CBOW analogy result ('government' - 'bank' + 'company') top 5:
department: 0.5511
sector: 0.5363
study: 0.5281
industry: 0.5202
moody: 0.4919

Skipgram analogy result ('government' - 'bank' + 'company') top 5:
sector: 0.5087
companies: 0.4836
cartel: 0.4795
sec: 0.4655
construction: 0.4624
